In [20]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('../experiments/results_v3.csv', parse_dates=['timestamp'])

SCORE_COLS = ['overall_score', 'structure', 'data_richness', 'sophistication', 'actionability', 'sentiment_balance']
SCORE_COLS = [c for c in SCORE_COLS if c in df.columns]
RADAR_COLS = [c for c in ['structure', 'data_richness', 'sophistication', 'actionability', 'sentiment_balance'] if c in df.columns]
MODE_COLORS = {'sequential': '#636EFA', 'group_chat': '#EF553B', 'single_agent': '#00CC96'}

print(f"Shape: {df.shape}")
print(f"Modes: {df['mode'].unique().tolist()}")
print(f"Instruments: {sorted(df['instrument'].unique().tolist())}")
print(f"Missing faithfulness: {df['faithfulness_score'].isna().sum()} / {len(df)}")
df.head(3)


Shape: (150, 17)
Modes: ['sequential', 'group_chat', 'single_agent']
Instruments: ['AAPL', 'BAC', 'CAT', 'JNJ', 'JPM', 'KO', 'MSFT', 'PFE', 'WMT', 'XOM']
Missing faithfulness: 11 / 150


,timestamp,instrument,sector,mode,provider,model,temperature,execution_time,overall_score,grade,structure,data_richness,sophistication,actionability,sentiment_balance,recommendation,faithfulness_score
0,2026-05-30 22:03:43.310193,AAPL,Technology,sequential,openai,gpt-4.1,0.2,85.35,79.16,B,70.7,92.0,78.9,85.0,57.1,BUY,1.000000
1,2026-05-30 22:05:57.362491,AAPL,Technology,sequential,openai,gpt-4.1,0.2,83.32,79.19,B,70.7,89.0,81.1,85.0,60.5,BUY,0.952381
2,2026-05-30 22:08:09.972527,AAPL,Technology,sequential,openai,gpt-4.1,0.2,77.62,71.13,B,72.0,67.5,81.7,70.0,59.1,BUY,1.000000


In [21]:
# ── Overall score: mean ± std per instrument ──────────────────────────────────
agg = df.groupby('instrument')['overall_score'].agg(['mean', 'std', 'count']).reset_index()
agg = agg.sort_values('mean', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=agg['instrument'], y=agg['mean'],
    error_y=dict(type='data', array=agg['std'].fillna(0)),
    marker_color=agg['mean'], marker_colorscale='RdYlGn',
    text=agg['mean'].round(1), textposition='outside'
))
fig.update_layout(title='Overall Score - mean ± std per instrument (all modes)', yaxis_range=[0, 105], height=450)
fig.show()


In [22]:
# ── Overall score distribution per instrument ─────────────────────────────────
fig = px.box(
    df.sort_values('instrument'), x='instrument', y='overall_score',
    color='mode', color_discrete_map=MODE_COLORS, points='all',
    title='Overall Score distribution per instrument, colored by mode'
)
fig.update_layout(showlegend=True, height=480)
fig.show()


In [23]:
# ── Overall score per mode ────────────────────────────────────────────────────
agg_mode = df.groupby('mode')['overall_score'].agg(['mean', 'std', 'count']).reset_index()

fig = px.bar(
    agg_mode, x='mode', y='mean', error_y='std',
    text='mean', color='mode', color_discrete_map=MODE_COLORS,
    title='Overall Score - mean ± std per mode'
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 105], height=420, showlegend=False)
fig.show()


In [24]:
# ── Heatmap: instrument × mode (mean overall_score) ───────────────────────────
if df['mode'].nunique() > 1:
    pivot = df.groupby(['instrument', 'mode'])['overall_score'].mean().unstack(fill_value=None)
    fig = px.imshow(
        pivot, text_auto='.1f', color_continuous_scale='RdYlGn',
        title='Mean Overall Score - instrument × mode', aspect='auto'
    )
    fig.update_layout(height=max(300, len(pivot) * 50 + 100))
    fig.show()


In [25]:
# ── Radar: sub-metrics per mode ───────────────────────────────────────────────
radar_df = df.groupby('mode')[RADAR_COLS].mean().reset_index()

fig = go.Figure()
for _, row in radar_df.iterrows():
    vals = list(row[RADAR_COLS]) + [row[RADAR_COLS[0]]]
    cats = RADAR_COLS + [RADAR_COLS[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats, fill='toself', name=row['mode'],
        line_color=MODE_COLORS.get(row['mode'])
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 100])),
    title='Sub-metric radar per mode (mean)', height=520
)
fig.show()


In [26]:
# ── Sub-metric heatmap: instrument × metric ───────────────────────────────────
metric_pivot = df.groupby('instrument')[RADAR_COLS].mean()
metric_pivot = metric_pivot.loc[
    df.groupby('instrument')['overall_score'].mean().sort_values(ascending=False).index
]
fig = px.imshow(
    metric_pivot, text_auto='.1f', color_continuous_scale='RdYlGn',
    title='Mean sub-metrics per instrument', aspect='auto'
)
fig.update_layout(height=max(300, len(metric_pivot) * 50 + 100))
fig.show()


In [27]:
# ── Execution time per instrument & mode ─────────────────────────────────────
fig = px.box(
    df.sort_values('instrument'), x='instrument', y='execution_time',
    color='mode', color_discrete_map=MODE_COLORS, points='all',
    title='Execution time (s) per instrument, colored by mode'
)
fig.update_layout(height=480)
fig.show()


In [28]:
# ── Grade distribution per mode ───────────────────────────────────────────────
if 'grade' in df.columns:
    grade_order = ['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D', 'F']
    grade_counts = df.groupby(['mode', 'grade']).size().reset_index(name='count')
    fig = px.bar(
        grade_counts, x='grade', y='count', color='mode',
        color_discrete_map=MODE_COLORS,
        barmode='group', category_orders={'grade': grade_order},
        title='Grade distribution per mode'
    )
    fig.update_layout(height=420)
    fig.show()


## Faithfulness (LLM-as-judge)

In [29]:
# ── Faithfulness score: mean per mode ─────────────────────────────────────────
faith_mode = df.dropna(subset=['faithfulness_score']).groupby('mode')['faithfulness_score'].agg(['mean', 'std', 'count']).reset_index()

fig = px.bar(
    faith_mode, x='mode', y='mean', error_y='std',
    text='mean', color='mode', color_discrete_map=MODE_COLORS,
    title='Faithfulness Score - mean ± std per mode (LLM-as-judge, DeepEval)'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 1.15], height=420, showlegend=False)
fig.show()


In [30]:
# ── Faithfulness score: distribution per instrument ───────────────────────────
fig = px.box(
    df.dropna(subset=['faithfulness_score']).sort_values('instrument'),
    x='instrument', y='faithfulness_score',
    color='mode', color_discrete_map=MODE_COLORS, points='all',
    title='Faithfulness Score distribution per instrument'
)
fig.update_layout(height=480, yaxis_range=[0, 1.05])
fig.show()


In [31]:
faith_pivot = df.dropna(subset=['faithfulness_score']).groupby(['instrument', 'mode'])['faithfulness_score'].mean().unstack(fill_value=None)
fig = px.imshow(
    faith_pivot, text_auto='.3f',
    color_continuous_scale='RdYlGn', zmin=0.8, zmax=1.0,
    title='Mean Faithfulness Score - instrument × mode', aspect='auto'
)
fig.update_layout(height=max(300, len(faith_pivot) * 50 + 100))
fig.show()


## Recommendation Analysis

In [32]:
rec_counts = df.groupby(['instrument', 'recommendation']).size().reset_index(name='count')
totals = rec_counts.groupby('instrument')['count'].transform('sum')
rec_counts['pct'] = (rec_counts['count'] / totals * 100).round(1)

fig = px.bar(
    rec_counts.sort_values('instrument'), x='instrument', y='pct',
    color='recommendation',
    color_discrete_map={'BUY': '#2ECC71', 'HOLD': '#F39C12', 'SELL': '#E74C3C'},
    text='pct', barmode='stack',
    title='Recommendation distribution per instrument (% of runs)'
)
fig.update_traces(texttemplate='%{text:.0f}%', textposition='inside')
fig.update_layout(height=450, yaxis_title='% of runs', yaxis_range=[0, 105])
fig.show()


In [33]:
rec_mode = df.groupby(['mode', 'recommendation']).size().reset_index(name='count')
totals_mode = rec_mode.groupby('mode')['count'].transform('sum')
rec_mode['pct'] = (rec_mode['count'] / totals_mode * 100).round(1)

fig = px.bar(
    rec_mode, x='mode', y='pct',
    color='recommendation',
    color_discrete_map={'BUY': '#2ECC71', 'HOLD': '#F39C12', 'SELL': '#E74C3C'},
    text='pct', barmode='stack',
    title='Recommendation distribution per mode'
)
fig.update_traces(texttemplate='%{text:.0f}%', textposition='inside')
fig.update_layout(height=420, yaxis_title='% of runs', yaxis_range=[0, 105])
fig.show()


## Sector Analysis

In [34]:
sector_agg = df.groupby('sector')['overall_score'].agg(['mean', 'std', 'count']).reset_index()
sector_agg = sector_agg.sort_values('mean', ascending=False)

fig = px.bar(
    sector_agg, x='sector', y='mean', error_y='std',
    text='mean', color='mean', color_continuous_scale='RdYlGn',
    title='Overall Score - mean ± std per sector'
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(height=440, yaxis_range=[0, 105], coloraxis_showscale=False)
fig.show()


In [35]:
faith_sector = df.dropna(subset=['faithfulness_score']).groupby('sector')['faithfulness_score'].agg(['mean', 'std']).reset_index()
faith_sector = faith_sector.sort_values('mean', ascending=False)

fig = px.bar(
    faith_sector, x='sector', y='mean', error_y='std',
    text='mean', color='mean', color_continuous_scale='RdYlGn',
    title='Faithfulness Score - mean ± std per sector'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(height=440, yaxis_range=[0, 1.15], coloraxis_showscale=False)
fig.show()


In [36]:
# ── Mean execution time per mode ─────────────────────────────────────────────
exec_mode = df.groupby('mode')['execution_time'].agg(['mean', 'std']).reset_index()
fig = px.bar(
    exec_mode, x='mode', y='mean', error_y='std',
    text='mean', color='mode', color_discrete_map=MODE_COLORS,
    title='Mean execution time (s) per mode'
)
fig.update_traces(texttemplate='%{text:.0f}s', textposition='outside')
fig.update_layout(height=400, showlegend=False)
fig.show()


## Summary Table

In [37]:
# ── Summary table: mode × instrument ─────────────────────────────────────────
summary = df.groupby(['mode', 'instrument']).agg(
    overall_score_mean=('overall_score', 'mean'),
    overall_score_std=('overall_score', 'std'),
    faithfulness_mean=('faithfulness_score', 'mean'),
    execution_time_mean=('execution_time', 'mean'),
    dominant_rec=('recommendation', lambda x: x.value_counts().idxmax()),
    n_runs=('overall_score', 'count'),
).round(2).reset_index()

summary.style \
    .background_gradient(subset=['overall_score_mean', 'faithfulness_mean'], cmap='RdYlGn') \
    .format({'overall_score_mean': '{:.1f}', 'overall_score_std': '±{:.1f}',
             'faithfulness_mean': '{:.3f}', 'execution_time_mean': '{:.0f}s'})


,mode,instrument,overall_score_mean,overall_score_std,faithfulness_mean,execution_time_mean,dominant_rec,n_runs
0,group_chat,AAPL,83.5,±3.6,0.960,257s,HOLD,5
1,group_chat,BAC,84.8,±1.4,0.980,290s,BUY,5
2,group_chat,CAT,80.9,±2.7,0.950,257s,HOLD,5
3,group_chat,JNJ,84.9,±2.7,0.950,277s,HOLD,5
4,group_chat,JPM,85.9,±2.6,0.960,292s,BUY,5
5,group_chat,KO,84.0,±4.4,0.960,241s,HOLD,5
6,group_chat,MSFT,85.3,±3.8,0.950,275s,BUY,5
7,group_chat,PFE,82.3,±5.0,0.920,312s,HOLD,5
8,group_chat,WMT,87.5,±5.1,0.980,254s,HOLD,5
9,group_chat,XOM,86.8,±1.6,0.930,270s,BUY,5
